# FP4 training and packed inference in Needle
Shreya Verma · Anshuman Mishra · Nidhi Prakash | Group 59 | CMU 10-414/714


## Research question
Which product in a linear layer is most sensitive to four-bit rounding? Does stochastic rounding recover accuracy? For batch-one inference, we measure how packing weights changes memory use and latency on the selected device.


## Format
NVFP4 uses signed E2M1 values with magnitudes 0, 0.5, 1, 1.5, 2, 3, 4 and 6. Each block of 16 shares an E4M3 scale. The tensor scale is its maximum absolute value divided by 6 × 448. Each block scale is its maximum magnitude divided by 6 and the tensor scale, then rounded to E4M3.

Between grid neighbors, stochastic rounding chooses the upper value with probability equal to the fractional distance. A normalized value of 0.2 rounds to 0.5 with probability 0.4. This preserves its expectation for fixed scales. NVFP4 training [1] motivates rounding gradients this way. MXFP4 [3, 6] provides the comparison with 32-value blocks and power-of-two scales.


## Implementation
Needle records tensor operations for automatic differentiation. A linear layer computes scores, and SGD updates its weights using the loss gradient. `QuantizedLinear` keeps the existing layer interface and calls `FP4Matmul`, whose switches independently round operands along each product’s reduction axis.

| Product | Formula | Reduction | Role |
|---|---|---|---|
| Fprop | XW | Input features | Compute activations |
| Dgrad | dY Wᵀ | Output features | Send gradients to earlier layers |
| Wgrad | Xᵀ dY | Batch | Compute the weight update |

Training uses a straight-through backward rule, FP32 master weights and FP32 matrix products on reconstructed operands. For inference, `NVFP4Matrix` packs two codes per byte plus scales. CPU and CUDA GEMV kernels decode the codes while multiplying by the input vector. The CUDA kernel reuses an input block across four output rows. Tests compare the native kernels against a NumPy reference.


## Run settings
Use a fresh Colab runtime and select **Run all**. Leave `FULL_STUDY = False` for the three-epoch, seed-0 demo (20,000 CIFAR-10 training images on CPU or 50,000 on GPU). Set it to `True` before starting to also run the registered study and diagnostics. Each execution writes a separate evidence archive.


In [ ]:
from pathlib import Path
import csv, hashlib, json, math, os, shutil, subprocess, sys, tarfile, tempfile, urllib.request, uuid, zipfile
from datetime import datetime, timezone
from IPython.display import display, Markdown, Image, FileLink

# AI-assisted report code: ChatGPT/Codex (GPT-6), adapting the team's Claude-assisted notebook.
FULL_STUDY = False
CODE_REF = 'c11a393db74e1752df7d4a5268e1ba2ae3b447ab'  # immutable implementation under evaluation
SEEDS = [0]
EPOCHS = 3
EXTRA_DIAGNOSTICS = FULL_STUDY
START = Path('/content') if Path('/content').is_dir() else Path(tempfile.gettempdir())
RUN = START / ('needle-results-' + uuid.uuid4().hex[:10])
RUN.mkdir()
checkout = Path(tempfile.mkdtemp(prefix='needle-source-')) / 'fp4-needle'
url = 'https://github.com/shreya-v7/fp4-needle.git'
subprocess.run(['git','clone','--quiet','--no-checkout',url,str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--quiet','--detach',CODE_REF],check=True)
def sha(ref):
    return subprocess.check_output(['git','-C',str(checkout),'rev-parse',ref],text=True).strip()
REVISION = sha('HEAD')
assert REVISION == sha(CODE_REF + '^{commit}')
print('Running commit:', REVISION, '| main at clone:', sha('origin/main'))
os.chdir(checkout)
def run(args, log=None):
    print(log or str(args[0]), flush=True)
    with (RUN / 'commands.jsonl').open('a') as audit:
        audit.write(json.dumps([str(x) for x in args]) + '\n')
    if log is None:
        subprocess.run([str(x) for x in args],check=True)
        return
    with (RUN / log).open('w') as stream:
        status = subprocess.run([str(x) for x in args], stdout=stream, stderr=subprocess.STDOUT)
    if status.returncode:
        print((RUN / log).read_text()[-2000:])
        raise RuntimeError(f'{log} failed. Partial evidence: {RUN}')
run([sys.executable,'-m','pip','install','-q','pybind11','cmake>=3.27,<4','numpy','pytest','matplotlib'])
cmake = shutil.which('cmake')
run([cmake,'-S','.','-B','build','-DPython_EXECUTABLE='+sys.executable])
run([cmake,'--build','build','--parallel','2'])
os.environ['PYTHONPATH'] = str(checkout/'python')
os.environ['NEEDLE_BACKEND'] = 'nd'
sys.path.insert(0,str(checkout/'python'))
import needle as ndl
import matplotlib.pyplot as plt
DEVICE = 'cuda' if ndl.cuda().enabled() else 'cpu'
gpu_info = ndl.cuda().device_info() if DEVICE == 'cuda' else None
SUBSET = 0 if DEVICE == 'cuda' else 20000
meta = dict(revision=REVISION, main_at_clone=sha('origin/main'), device=DEVICE, gpu=gpu_info, seeds=SEEDS,
            full_study=FULL_STUDY,
            registered=dict(dataset='CIFAR-10', train_examples=50000, test_examples=10000,
                            main_seeds=list(range(5)), epochs=20, lr_seeds=[0,1], depth_seeds=[0,1,2],
                            sweep_epochs=10, grad_checkpoints=[0,391,1173], grad_seed=0, sr_draws=32),
            gemv_seed='rows * 100003 + cols', mnist=dict(seed=0, train_examples=60000, test_examples=10000, epochs=3),
            cifar_test_examples=10000, epochs=EPOCHS, train_examples=SUBSET or 50000, diagnostics=EXTRA_DIAGNOSTICS,
            utc=datetime.now(timezone.utc).isoformat())
(RUN/'run.json').write_text(json.dumps(meta,indent=2))
print(f'{DEVICE} | demo seed {SEEDS}, {EPOCHS} epochs, {SUBSET or 50000:,} training images')

print('Run directory:', RUN)
if DEVICE == 'cpu': print('CUDA tests and GPU timings skipped. Benchmark device: CPU.')


## Correctness
The FP4 tests check rounding, packing, gradients and kernel outputs against the reference. Pytest reports any device-specific skips.


In [ ]:
run([sys.executable,'-m','pytest','tests/fp4','-q','-rs'],'tests.txt')

print((RUN/'tests.txt').read_text().strip().splitlines()[-1])


### Data
Verify the CIFAR-10 and MNIST input files against their checksums.


In [ ]:
data = checkout/'data'
data.mkdir(exist_ok=True)
def fetch(url, name, digest):
    path = data/name
    if not path.exists(): urllib.request.urlretrieve(url,path)
    with path.open('rb') as f:
        actual=hashlib.file_digest(f,'md5').hexdigest()
    if actual != digest: raise ValueError(f'Unexpected data file: {name}')
    return path
cifar = fetch('https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz','cifar-10-python.tar.gz','c58f30108f718f92721af3b95e74349a')
with tarfile.open(cifar) as archive: archive.extractall(data,filter='data')
MNIST = {"train-images-idx3-ubyte.gz": "0cb5796cd8e7c7bffef03edbdb01fac5",
         "train-labels-idx1-ubyte.gz": "23e0cdb926d142c2d470a3641a19d1ff",
         "t10k-images-idx3-ubyte.gz":  "3d4db374c734e4f92d346e063bf6e41a",
         "t10k-labels-idx1-ubyte.gz":  "0407fd84edb9a00bbad73c377ff95de9"}

for name, digest in MNIST.items():
    fetch('https://raw.githubusercontent.com/dlsyscourse/hw0/main/data/'+name,name,digest)
print('Training and test data ready.')


### Training ablation
Hold initialization, batch order, architecture and optimizer fixed. Compare each placement with FP32, then compare stochastic with nearest rounding at the same placement. Differences are in test-accuracy percentage points.


In [ ]:
CONFIGS = ['fp32','fprop','dgrad','dgrad+sr','wgrad','wgrad+sr','fprop+dgrad+wgrad','fprop+dgrad+wgrad+sr']
final = {}
for seed in SEEDS:
    path = RUN/f'cifar_s{seed}.csv'
    args = [sys.executable,'apps/fp4_experiment.py','--dataset','cifar10','--configs','all','--only',*CONFIGS,
            '--device',DEVICE,'--epochs',EPOCHS,'--seed',seed,'--train-subset',SUBSET,'--out',path,
            '--save-weights',str(RUN/'ckpt'/'cifar_{config}_s{seed}.npz')]
    run(args, f'training-s{seed}.txt')
    with path.open() as f:
        observations = [r for r in csv.DictReader(f) if r['split'] == 'test']
    for name in CONFIGS:
        epochs = [int(r['epoch']) for r in observations if r['config'] == name and int(r['seed']) == seed]
        print(name, 'seed', seed, 'completed epochs', epochs)
        if epochs != list(range(1, EPOCHS + 1)):
            raise RuntimeError(f'Partial demo for {name}; see {path}. No comparison available.')
    rows = [r for r in observations if int(r['epoch']) == EPOCHS]
    by_name = {r['config']:100*float(r['accuracy']) for r in rows}
    if len(rows)!=len(CONFIGS) or set(by_name)!=set(CONFIGS) or any(not math.isfinite(v) for v in by_name.values()):
        raise RuntimeError(f'Incomplete or invalid final epoch for seed {seed}; inspect {path}')
    final[seed]=by_name


In [ ]:
from statistics import mean, pstdev
summary=[]
for name in CONFIGS:
    values=[final[s][name] for s in SEEDS]
    delta=mean(final[s][name]-final[s]['fp32'] for s in SEEDS)
    summary.append((name,mean(values),pstdev(values) if len(values)>1 else None,delta))
print('Device', DEVICE, '| seed(s)', SEEDS, '| examples', SUBSET or 50000, '| final epoch', EPOCHS)
print('configuration              accuracy (%)    seed SD (pp)     paired gap vs FP32 (pp)')
for name,avg,sd,delta in summary:
    sd_text=f'{sd:.2f}' if sd is not None else 'single seed'
    print(f'{name:27s} {avg:9.2f} {sd_text:>14s} {delta:+18.2f}')
for name in ['dgrad','wgrad','fprop+dgrad+wgrad']:
    shift=mean(final[s][name+'+sr']-final[s][name] for s in SEEDS)
    print(f'{name} · SR minus nearest: {shift:+.2f} percentage points')
fig,ax=plt.subplots(figsize=(9,4))
ax.barh([x[0] for x in summary],[x[3] for x in summary],color='#257b78')
ax.axvline(0,color='black',lw=1); ax.set_xlabel('Final test accuracy minus paired FP32 baseline (percentage points)')
ax.invert_yaxis(); fig.tight_layout(); fig.savefig(RUN/'training.png'); plt.show()


### GEMV benchmark
Compare dense and packed weights at the same shape on the same device. Dense time divided by packed time above one favors the packed kernel. Weight bytes include scales and padding.


In [ ]:
sizes = [1024,2048,4096] + ([8192] if DEVICE=='cuda' else [])
run([sys.executable,'benchmarks/bench_gemv.py','--device',DEVICE,'--sizes',*sizes,*(['--shapes','11008x4096','4096x11008'] if DEVICE=='cuda' else []),'--out',RUN/'gemv.csv'],'gemv.txt')
with (RUN/'gemv.csv').open() as f: rows = {r['kernel']:r for r in csv.DictReader(f) if r['shape']=='4096x4096'}
a,b = rows['fp32 gemv (ours)'],rows['nvfp4 gemv (ours)']
print(f"Device {DEVICE} · 4096 × 4096 · dense {float(a['time_us']):.2f} µs · packed {float(b['time_us']):.2f} µs")
print(f"Dense/packed time: {float(a['time_us'])/float(b['time_us']):.3f}× · dense/packed weight bytes: {float(a['weight_MiB'])/float(b['weight_MiB']):.3f}×")
display(Image(filename=str(RUN/'gemv.png')))


### MNIST inference
Use the same trained weights for both models. Test accuracy uses batched evaluation with reconstructed weights. Single-image timing calls the packed kernels directly.


In [ ]:
run([sys.executable,'apps/fp4_inference.py','--device',DEVICE],'inference.txt')

print((RUN/'inference.txt').read_text().strip())


### Weight formats
Quantize the same FP32 checkpoint with NVFP4 and MXFP4. Higher SQNR means lower relative reconstruction error. Compare test accuracy separately.


In [ ]:
ckpt = RUN/'ckpt'/f'cifar_fp32_s{SEEDS[0]}.npz'
run([sys.executable,'apps/fp4_mxfp4.py',ckpt,'--device',DEVICE,'--out',RUN/'formats.md'])
display(Markdown((RUN/'formats.md').read_text()))


### Mechanism checks
At fixed weights and input, compare weight gradients against FP32 across rounding modes. The second experiment varies how weights are quantized between forward and backward passes.


In [ ]:
if EXTRA_DIAGNOSTICS:
    run([sys.executable,'apps/fp4_grad_error.py','--dataset','cifar10','--device',DEVICE,
         '--checkpoints',0,391,1173,'--out',RUN/'gradients.csv'],'gradients.txt')
    for seed in SEEDS:
        run([sys.executable,'apps/fp4_experiment.py','--dataset','cifar10','--device',DEVICE,'--configs','weights',
             '--epochs',EPOCHS,'--seed',seed,'--train-subset',SUBSET,'--out',RUN/f'weights_s{seed}.csv'],f'weights_s{seed}.txt')
else:
    print('Optional gradient and weight experiments skipped in this run.')


In [ ]:
if EXTRA_DIAGNOSTICS:
    print('Gradient relative errors, first hidden layer; mean column uses the configured draw count')
    with (RUN/'gradients.csv').open() as f:
        for r in csv.DictReader(f):
            if r['layer']=='layer1' and r['config'] in ('dgrad','dgrad+sr'):
                print(r['step'],r['config'],'one draw',r['rel_error'],'mean',r['rel_error_of_mean'],'draws',r['sr_draws'])
    print('Weight-mode experiment, final-epoch accuracy (%)')
    for seed in SEEDS:
        with (RUN/f'weights_s{seed}.csv').open() as f:
            rows=[r for r in csv.DictReader(f) if r['split']=='test' and int(r['epoch'])==EPOCHS]
        if len({r['config'] for r in rows})!=4: raise RuntimeError(f'Incomplete weight-mode run for seed {seed}')
        for r in rows: print(seed,r['config'],round(100*float(r['accuracy']),2))


### Preregistered CIFAR-10 experiment
Metric A averages held-out accuracy in the last three scheduled epochs. Subtract metric A within each seed before computing its mean, sample SD and paired standard error. The main study fixes width 256, two hidden layers, batch 128, SGD momentum 0.9, learning rate 0.05, all 50,000 training images and seeds 0 through 4. Each configuration gets 20 epochs. A separate random stream supplies SR. The two Dgrad operand controls keep either W or dY in FP32. Their comparison distinguishes a changed learning signal from the effect of regrouping the weights.

H1 requires SR to beat nearest by more than two paired standard errors, with at least four positive seeds for each of Dgrad and all-three. H1b requires both SR means to be within one percentage point of FP32. H2 requires rounding dY alone to lose more than one point and two paired standard errors, while W-only and dY-only with SR each remain within one point of FP32. H3 compares the mean FP32-minus-Dgrad gap at epochs 3 and 20. An equal or larger final gap counts as genuine failure, a final gap below half the early gap as slower learning, and intermediate values as partial closure.

The sensitivity runs change only learning rate (0.01 or 0.1, seeds 0 and 1) or depth (one or four hidden layers, seeds 0 through 2). They use ten epochs, scoring epochs 8–10. The matching default setting uses the first ten epochs of the main study. The directional criterion is a positive SR gain exceeding two paired standard errors. An observed non-finite training loss counts as chance from that epoch forward. Interrupted jobs and missing measurements receive a partial status. The original registered rules are in `docs/PREREGISTRATION.md`.


In [ ]:
from collections import defaultdict
import shlex
sys.path.insert(0, str(checkout / 'apps'))
from fp4_stats import main_report, sweep_report, plot_curves

def checked_series(path, configs, seed, horizon):
    scores = defaultdict(list)
    diverged = {}
    if path.exists():
        with path.open() as stream:
            for row in csv.DictReader(stream):
                name, epoch = row['config'], int(row['epoch'])
                if name not in configs or int(row['seed']) != seed:
                    raise ValueError(f'Unexpected identity in {path}')
                if row['split'] == 'train' and not math.isfinite(float(row['loss'])):
                    diverged[name] = epoch
                if row['split'] == 'test':
                    accuracy = float(row['accuracy'])
                    if not 0 <= accuracy <= 1 or not math.isfinite(float(row['loss'])):
                        raise ValueError(f'Invalid test row in {path}')
                    scores[name].append((epoch, 100 * accuracy))
    result = {}
    for name in configs:
        rows = scores[name]
        end = diverged.get(name, horizon + 1)
        observed = [e for e, _ in rows]
        if not 1 <= end <= horizon + 1 or observed != list(range(1, end)):
            raise ValueError(f'Partial or duplicate epochs in {path.name}: {name}, seed {seed}, observed epochs {observed}')
        values = [a for _, a in rows]
        if name in diverged:
            print(f'{name} diverged in epoch {end}; assigning chance through epoch {horizon}.')
            values += [10.0] * (horizon - end + 1)
        result[name, seed] = values
    return result

if FULL_STUDY:
    study = RUN / 'extended'
    study.mkdir(exist_ok=False)
    groups = defaultdict(dict)
    # The versioned job list sets the configurations, seeds, depth, lr and epoch count.
    lines = (checkout / 'scripts/extended_jobs.txt').read_text().splitlines()
    (study / 'planned-jobs.txt').write_text('\n'.join(lines))
    for line in lines:
        args = shlex.split(line)
        output = study / Path(args[args.index('--out') + 1]).name
        args[args.index('--out') + 1] = str(output)
        seed = int(args[args.index('--seed') + 1])
        horizon = int(args[args.index('--epochs') + 1])
        configs = (args[args.index('--only')+1:args.index('--epochs')] if '--only' in args
                   else ['fp32','dgrad','dgrad+sr'])
        command = [sys.executable,'apps/fp4_experiment.py','--dataset','cifar10','--device',DEVICE,*args]
        run(command, f'extended/{output.stem}.log')
        groups[output.stem.split('_seed')[0]].update(checked_series(output, configs, seed, horizon))
    main = groups['main']
    text = main_report(main)
    for prefix, default, label in [('lr','0.05','Learning rate'),('depth','2','Hidden depth')]:
        sweep = {k[len(prefix):]: v for k,v in groups.items() if k.startswith(prefix)}
        text += sweep_report(label, sweep, main, default)
    report = '\n'.join(text)
    (study / 'verdicts.md').write_text(report)
    display(Markdown(report))
    plot_curves(main, study / 'curves.png')
    display(Image(filename=str(study / 'curves.png')))
else:
    print('Skipped full study: zero registered jobs executed; H1, H1b, H2, H3 and sensitivity unscored.')


## Interpretation and limits
Use the isolated products to identify sensitive placements. An SR gain can coexist with a remaining gap to FP32. The demo has one seed. Use the registered paired analysis for conclusions across initializations.

Gradient error measures distortion on a fixed batch. Accuracy measures its accumulated effect during learning. Packed bytes measure weight storage, while latency includes decode overhead and depends on device and shape. The results apply to these MLPs and simulated FP4 training. Native FP4 tensor-core throughput and language-model decoding require separate measurements.


## References


1. NVIDIA. [Pretraining Large Language Models with NVFP4](https://arxiv.org/abs/2509.25149). 2025.
2. Chmiel, Fishman, Banner, Soudry. [FP4 All the Way: Fully Quantized Training of LLMs](https://arxiv.org/abs/2505.19115). 2025.
3. Tseng, Yu, Park. [Training LLMs with MXFP4](https://arxiv.org/abs/2502.20586). 2025.
4. Castro et al. [Quartet: Native FP4 Training Can Be Optimal for Large Language Models](https://arxiv.org/abs/2505.14669). 2025.
5. DeepSeek-AI. [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437). 2024.
6. Open Compute Project. OCP Microscaling Formats (MX) Specification v1.0. 2023.
7. Datasets: MNIST (LeCun et al., 1998); CIFAR-10 (Krizhevsky, 2009).
8. Cook and Thinking Machines Lab. [End-to-End 4-Bit Training](https://thinkingmachines.ai/blog/end-to-end-4-bit-training/). 2026.


## AI disclosure
Anthropic Claude Code (Opus 5.5) assisted with the FP4 Python and native backend code, experiment scripts, tests, CI, literature search and drafting. Source comments identify assisted sections. The starting Needle code came from course homework. OpenAI ChatGPT/Codex (GPT-6) assisted with the notebook, README, syntax checks and output-provenance checks. The run archive contains the test log, commands and measurements for team review.


In [ ]:
# Export only tracked source and tests at the exact evaluated revision.
source_zip = START / f'fp4-needle-{REVISION[:12]}-source.zip'
subprocess.run(['git','archive','--format=zip',f'--output={source_zip}',REVISION],check=True)
print('Source archive:', source_zip)


## Run evidence
Download the results ZIP before disconnecting. It includes settings, logs, CSVs and figures.


In [ ]:
archive = START/(RUN.name+'.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(RUN.rglob('*')):
        if path.is_file() and path.suffix != '.npz': z.write(path,path.relative_to(RUN))
print('Results:',archive)
os.chdir(START)
display(FileLink(archive.name))
